In [2]:
## Neural Network Implementation (Before Optimization)

using Random
using Statistics
using LinearAlgebra
using Printf


# Layer_dense (Dense Layer) 

In [3]:
# Set a seed for reproducibility of random weights
Random.seed!(42)

# --- Layer_dense (Dense Layer) ---
mutable struct Layer_dense
    inputs::Matrix{Float32}
    weights::Matrix{Float32} 
    biases ::Matrix{Float32}
    output::Matrix{Float32} 
    dweights::Matrix{Float32}
    dbiases::Matrix{Float32}
    dinputs::Matrix{Float32}
        # Add momentums
    weight_momentums::Matrix{Float32}
    bias_momentums::Matrix{Float32}
    function Layer_dense(n_inputs::Int, n_neurons::Int)
        # Initialize weights with small random numbers from a Gaussian distribution
        weights = Float32(0.01) * randn(Float32, n_inputs, n_neurons)
        biases = zeros(Float32, 1, n_neurons)
            weight_momentums = zeros(Float32, n_inputs, n_neurons)
        bias_momentums = zeros(Float32, 1, n_neurons)
        new(Matrix{Float32}(undef,0,0), weights, biases, Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), weight_momentums, bias_momentums)
    end
end

# Forward Pass for dense layer
function forward(layer::Layer_dense, inputs::Matrix{Float32})
    # Store inputs for use in the backward pass
    layer.inputs = inputs 
    # Calculate output: Z = XW + B
    layer.output = inputs * layer.weights .+ layer.biases
end

# Backward Pass for dense layer
function backward(layer::Layer_dense, upstream_gradient::Matrix{Float32})
    # Gradient of the loss with respect to weights: dL/dW = X^T * dL/dZ
    layer.dweights = layer.inputs' * upstream_gradient
    # Gradient of the loss with respect to biases: dL/dB = sum(dL/dZ)
    layer.dbiases = sum(upstream_gradient, dims=1)
    # Gradient of the loss with respect to inputs: dL/dX = dL/dZ * W^T
    layer.dinputs = upstream_gradient * layer.weights'
end

backward (generic function with 1 method)

# Activation_ReLU (ReLU Activation Function)

In [4]:
# --- Activation_ReLU (ReLU Activation Function) ---
mutable struct Activation_ReLU
    inputs::Matrix{Float32}
    output::Matrix{Float32}
    dinputs::Matrix{Float32}
    Activation_ReLU() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

# Forward pass for ReLU
function forward(activation::Activation_ReLU, inputs::Matrix{Float32})
    # Store inputs for the backward pass
    activation.inputs = inputs
    # Apply ReLU: max(0, input)
    activation.output = max.(Float32(0.0), inputs)
end

# Backward pass for ReLU
function backward(activation::Activation_ReLU, upstream_gradient::Matrix{Float32})
    # Start with a copy of the upstream gradient
    activation.dinputs = copy(upstream_gradient)
    # Zero out gradients where the original input was non-positive
    activation.dinputs[activation.inputs .<= 0] .= Float32(0.0)
end

backward (generic function with 2 methods)

# Activation_Softmax_Loss_CategoricalCrossentropy

In [5]:
mutable struct Activation_Softmax_Loss_CategoricalCrossentropy
    output::Matrix{Float32} # This will store the softmax probabilities
    dinputs::Matrix{Float32}
    
    Activation_Softmax_Loss_CategoricalCrossentropy() = new(Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0))
end

function forward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, inputs::Matrix{Float32}, y_true::Vector{Int})
    # 1. Softmax Activation
    exp_values = exp.(inputs .- maximum(inputs, dims=2))
    probabilities = exp_values ./ sum(exp_values, dims=2)
    combo.output = probabilities

    # 2. Categorical Cross-Entropy Loss Calculation
    n_samples = size(probabilities, 1)
    # Clip data to prevent division by 0
    probabilities_clipped = clamp.(probabilities, Float32(1e-7), Float32(1.0) - Float32(1e-7))
    
    # Get the probabilities corresponding to the true labels
    correct_confidences = [probabilities_clipped[i, y_true[i]] for i in 1:n_samples]
    
    # Calculate negative log likelihoods and return the average loss
    negative_log_likelihoods = -log.(correct_confidences)
    data_loss = mean(negative_log_likelihoods)
    return data_loss
end

function backward(combo::Activation_Softmax_Loss_CategoricalCrossentropy, y_true::Vector{Int})
    n_samples = size(combo.output, 1)
    n_outputs = size(combo.output, 2)

    # Copy probabilities
    combo.dinputs = copy(combo.output)

    # Calculate gradient
    # For each sample, subtract 1 from the probability of the true class
    for i in 1:n_samples
        combo.dinputs[i, y_true[i]] -= Float32(1.0)
    end

    # Normalize gradient by the number of samples
    combo.dinputs ./= n_samples
end

backward (generic function with 3 methods)

# Data Generation Function

In [6]:
function create_data(n_points::Int, n_classes::Int)
    X = zeros(Float32, n_points * n_classes, 2) # Features
    y = zeros(Int, n_points * n_classes)        # Labels

    for class_number in 0:(n_classes - 1)
        ix = (class_number * n_points + 1):((class_number + 1) * n_points)
        
        # Radius for the spiral
        r = range(Float32(0.0), Float32(1.0), length=n_points)
        
        # Angle for the spiral
        t = range(class_number * Float32(4.0), (class_number + 1) * Float32(4.0), length=n_points) .+ (randn(Float32, n_points) * Float32(0.2))
        
        # Calculate x and y coordinates
        X[ix, 1] = r .* sin.(t * Float32(2.5))
        X[ix, 2] = r .* cos.(t * Float32(2.5))
        
        # Assign class labels (Julia is 1-indexed, so add 1)
        y[ix] .= class_number + 1
    end
    return X, y
end

create_data (generic function with 1 method)

# Main Execution

In [7]:
# Generate the spiral dataset
X, y = create_data(100, 3)
println("--- Dataset Generated ---")
println("Shape of X: ", size(X))
println("Shape of y: ", size(y))
println("First 5 samples of X:\n", X[1:5,:])
println("First 5 labels of y:\n", y[1:5])

# Define the network architecture
layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
layer2 = Layer_dense(64, size(unique(y), 1)) # Number of unique classes in y
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

println("\n--- Network Architecture Initialized ---")
println("Layer 1 Weights shape: ", size(layer1.weights))
println("Layer 1 Biases shape: ", size(layer1.biases))
println("Layer 2 Weights shape: ", size(layer2.weights))
println("Layer 2 Biases shape: ", size(layer2.biases))

# --- Training Loop (10 Epochs - No Optimization) ---
println("\n--- Starting Training Loop (10 Epochs - No Optimization) ---")
println("Note: Weights and biases will NOT be updated in this loop.")
println("      Loss and accuracy will remain constant as there's no optimizer.")

n_epochs = 10

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(layer1, X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss = forward(loss_activation, layer2.output, y)
    
    # Calculate accuracy
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    # Backward pass for Softmax + Categorical Cross-entropy
    backward(loss_activation, y)
    
    # Backward pass for Layer 2 (Dense)
    backward(layer2, loss_activation.dinputs)
    
    # Backward pass for Activation 1 (ReLU)
    backward(activation1, layer2.dinputs)
    
    # Backward pass for Layer 1 (Dense)
    backward(layer1, activation1.dinputs)

    # --- Print Results for the Epoch ---
    @printf "Epoch %d: Loss = %.4f, Accuracy = %.4f\n" epoch data_loss accuracy
    
    # Optional: Print some gradients to verify they are being calculated
    # if epoch == 1 || epoch == n_epochs
    #     println("  Layer 1 dWeights (first 2x2): \n", layer1.dweights[1:min(2, size(layer1.dweights,1)), 1:min(2, size(layer1.dweights,2))])
    #     println("  Layer 2 dBiases: ", layer2.dbiases)
    # end
end


--- Dataset Generated ---
Shape of X: (300, 2)
Shape of y: (300,)
First 5 samples of X:
First 5 samples of X:
Float32[0.0 0.0; -0.0033582626 0.009526409; -0.0047014733 0.019647334; -0.0019258887 0.03024177; 0.0021381031 0.04034743]
First 5 labels of y:
Float32[0.0 0.0; -0.0033582626 0.009526409; -0.0047014733 0.019647334; -0.0019258887 0.03024177; 0.0021381031 0.04034743]
First 5 labels of y:
[1, 1, 1, 1, 1]
[1, 1, 1, 1, 1]

--- Network Architecture Initialized ---
Layer 1 Weights shape: (2, 64)
Layer 1 Biases shape: (1, 64)
Layer 2 Weights shape: (64, 3)
Layer 2 Biases shape: (1, 3)

--- Starting Training Loop (10 Epochs - No Optimization) ---
Note: Weights and biases will NOT be updated in this loop.
      Loss and accuracy will remain constant as there's no optimizer.
Epoch 1: Loss = 1.0986, Accuracy = 0.3967
Epoch 2: Loss = 1.0986, Accuracy = 0.3967
Epoch 3: Loss = 1.0986, Accuracy = 0.3967
Epoch 4: Loss = 1.0986, Accuracy = 0.3967
Epoch 5: Loss = 1.0986, Accuracy = 0.3967
Epoch 6:

## Lets implement the OPTIMIZER_SGD


In [8]:
mutable struct Optimizer_SGD
    learning_rate::Float32
    function Optimizer_SGD(learning_rate::Float32=Float32(1.0))
        new(learning_rate)
    end
end
# Update parameters for a dense layer using SGD
function update_parameters(optimizer::Optimizer_SGD, layer::Layer_dense)
    layer.weights .+= -optimizer.learning_rate .* layer.dweights
    layer.biases .+= -optimizer.learning_rate .* layer.dbiases
end

update_parameters (generic function with 1 method)

In [9]:
# Define the network architecture
layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
layer2 = Layer_dense(64, size(unique(y), 1)) # Number of unique classes in y
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Define the optimizer
optimizer = Optimizer_SGD(Float32(0.5)) # Using a learning rate of 0.5

println("\n--- Network Architecture Initialized ---")
println("Layer 1 Weights shape: ", size(layer1.weights))
println("Layer 1 Biases shape: ", size(layer1.biases))
println("Layer 2 Weights shape: ", size(layer2.weights))
println("Layer 2 Biases shape: ", size(layer2.biases))
println("Optimizer Learning Rate: ", optimizer.learning_rate)


# --- Training Loop (With SGD Optimization) ---
println("\n--- Starting Training Loop (With SGD Optimization) ---")
println("Note: Weights and biases will now be updated, so loss and accuracy should change.")

n_epochs = 10000 # Changed to 10000 as per your previous request

# Initialize data_loss and accuracy outside the loop
data_loss::Float32 = Float32(0.0)
accuracy::Float32 = Float32(0.0)

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(layer1, X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss = forward(loss_activation, layer2.output, y)
    
    # Calculate accuracy
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    backward(loss_activation, y)
    backward(layer2, loss_activation.dinputs)
    backward(activation1, layer2.dinputs)
    backward(layer1, activation1.dinputs)

    # --- Update Parameters (Optimization Step) ---
    update_parameters(optimizer, layer1)
    update_parameters(optimizer, layer2)

    # --- Print Results for the Epoch ---
    if epoch == 1 || epoch % 100 == 0 # Print for the first epoch and every 100th epoch
        @printf "Epoch %d: Loss = %.4f, Accuracy = %.4f\n" epoch data_loss accuracy
    end
end

println("\n--- Training Loop Complete ---")
println("Final Loss: ", data_loss)
println("Final Accuracy: ", accuracy)
println("\nNow you can see the loss decreasing and accuracy increasing over epochs!")


--- Network Architecture Initialized ---
Layer 1 Weights shape: (2, 64)
Layer 1 Biases shape: (1, 64)
Layer 2 Weights shape: (64, 3)
Layer 2 Biases shape: (1, 3)
Optimizer Learning Rate: 0.5

--- Starting Training Loop (With SGD Optimization) ---
Note: Weights and biases will now be updated, so loss and accuracy should change.
Epoch 1: Loss = 1.0986, Accuracy = 0.3300
Epoch 100: Loss = 1.0979, Accuracy = 0.4033
Epoch 100: Loss = 1.0979, Accuracy = 0.4033
Epoch 200: Loss = 1.0921, Accuracy = 0.4100
Epoch 300: Loss = 1.0815, Accuracy = 0.4300
Epoch 400: Loss = 1.0771, Accuracy = 0.4333
Epoch 500: Loss = 1.0754, Accuracy = 0.4267
Epoch 600: Loss = 1.0744, Accuracy = 0.4367
Epoch 700: Loss = 1.0732, Accuracy = 0.4400
Epoch 800: Loss = 1.0721, Accuracy = 0.4333
Epoch 900: Loss = 1.0709, Accuracy = 0.4367
Epoch 1000: Loss = 1.0698, Accuracy = 0.4433
Epoch 1100: Loss = 1.0686, Accuracy = 0.4367
Epoch 1200: Loss = 1.0674, Accuracy = 0.4267
Epoch 200: Loss = 1.0921, Accuracy = 0.4100
Epoch 300

# 2. The Exploration-Exploitation Strategy
### Learning Rate Decay formalizes an exploration-exploitation strategy to counteract these issues.
• Exploration (Initial Stage): The training should begin with a high learning rate (large step size) to explore the entire loss landscape quickly and efficiently.

• Exploitation (Later Stage): Once the optimization has explored the landscape and found an area close to the potential minimum, the step size should be gradually reduced (decayed). This reduction prevents oscillations and allows the optimization to converge smoothly and accurately onto the global minimum

$$\alpha_T = \frac{\alpha_0}{1 + \text{Decay} \times T}$$

• $Role of Iteration$ ($T$): The iteration number ($T$) is in the denominator, ensuring that as $T$ increases, the overall step size ($\alpha_T$) decreases.

•$ Role of Decay Parameter (DK)$: The decay parameter controls how fast the step size decreases. If the DK parameter is set too high, the learning rate can become very small too early, potentially causing the optimization to get stuck in local minima. Typical practical values for the decay rate are around $0.001$ or $10^{-3}$

Let's break down the `Optimizer_SGD` constructor and how the learning rate decay works with some mock data.

First, let's clarify the `Optimizer_SGD` struct with the decay parameters:



In [10]:
mutable struct Optimizer_SGD_With_Decay
    learning_rate::Float32          # The initial learning rate (alpha_0)
    decay::Float32                  # The decay rate (DK)
    iterations::Int                 # The current iteration number (T)
    current_learning_rate::Float32  # The learning rate for the current iteration (alpha_T)

    # Constructor
    function Optimizer_SGD_With_Decay(learning_rate::Float32=Float32(1.0), decay::Float32=Float32(0.0))
        # Initialize iterations to 0
        # Initially, current_learning_rate is the same as the initial learning_rate
        new(learning_rate, decay, 0, learning_rate)
    end
end



**Explanation with Mock Data:**

Imagine you create an optimizer like this:



In [11]:
optimizer = Optimizer_SGD_With_Decay(Float32(0.1), Float32(0.001))

Optimizer_SGD_With_Decay(0.1f0, 0.001f0, 0, 0.1f0)



Here's what happens inside the constructor:

*   `learning_rate` (initial $\alpha_0$) is set to `0.1`. This is your starting learning rate.
*   `decay` (DK) is set to `0.001`. This controls how fast the learning rate will decrease.
*   `iterations` (T) is initialized to `0`. This counter will increase with each training step.
*   `current_learning_rate` (initial $\alpha_T$) is also initialized to `0.1`. At the very beginning (iteration 0), there's no decay applied yet.

**How `current_learning_rate` changes over iterations:**

The core idea is that `current_learning_rate` will be updated *before* each parameter update step using the formula:

$$\alpha_T = \frac{\alpha_0}{1 + \text{Decay} \times T}$$

Let's trace it for a few mock iterations:

**Before Epoch 1 (T=0):**
*   `optimizer.learning_rate` = `0.1`
*   `optimizer.decay` = `0.001`
*   `optimizer.iterations` = `0`
*   `optimizer.current_learning_rate` = `0.1` (from constructor)

**During Epoch 1:**

1.  **`pre_update_parameters(optimizer)` is called:**
    *   Since `optimizer.decay` (`0.001`) is greater than 0, the formula is applied:
    *   `current_learning_rate` = `0.1 / (1 + 0.001 * 0)`
    *   `current_learning_rate` = `0.1 / (1 + 0)`
    *   `current_learning_rate` = `0.1`
    *   So, for the first epoch, the learning rate used is `0.1`.

2.  **`update_parameters(optimizer, layer)` is called:**
    *   The weights and biases are updated using `optimizer.current_learning_rate` (`0.1`).

3.  **`post_update_parameters(optimizer)` is called:**
    *   `optimizer.iterations` is incremented: `optimizer.iterations` becomes `1`.

**During Epoch 2:**

1.  **`pre_update_parameters(optimizer)` is called:**
    *   `optimizer.iterations` is now `1`.
    *   `current_learning_rate` = `0.1 / (1 + 0.001 * 1)`
    *   `current_learning_rate` = `0.1 / 1.001`
    *   `current_learning_rate` $\approx$ `0.0999` (slightly less than 0.1)

2.  **`update_parameters(optimizer, layer)` is called:**
    *   The weights and biases are updated using `optimizer.current_learning_rate` ($\approx$ `0.0999`).

3.  **`post_update_parameters(optimizer)` is called:**
    *   `optimizer.iterations` is incremented: `optimizer.iterations` becomes `2`.

**During Epoch 100:**

1.  **`pre_update_parameters(optimizer)` is called:**
    *   `optimizer.iterations` is now `99`.
    *   `current_learning_rate` = `0.1 / (1 + 0.001 * 99)`
    *   `current_learning_rate` = `0.1 / (1 + 0.099)`
    *   `current_learning_rate` = `0.1 / 1.099`
    *   `current_learning_rate` $\approx$ `0.09099` (noticeably smaller than 0.1)

As you can see, with each passing epoch (as `iterations` increases), the denominator `(1 + Decay * T)` gets larger, causing the `current_learning_rate` to gradually decrease. This allows the model to take larger steps initially to explore the loss landscape and then smaller, more precise steps as it approaches a minimum.

 let's implement the `pre_update_parameters` and `post_update_parameters` functions and then modify the `update_parameters` function, explaining each with mock data.

### 1. `pre_update_parameters` Function

This function is responsible for calculating the `current_learning_rate` based on the initial learning rate, decay, and current iteration. It should be called *before* the actual weight and bias updates in each training step.



In [12]:
function pre_update_parameters(optimizer::Optimizer_SGD_With_Decay)
 # Only apply decay if a decay rate is set (i.e., > 0)
    if optimizer.decay>0
        optimizer.current_learning_rate=optimizer.learning_rate/(Float32(1.0)+optimizer.decay*optimizer.iterations)
    end
end

pre_update_parameters (generic function with 1 method)



**Explanation with Mock Data for `pre_update_parameters`:**

Let's assume you have an `optimizer` instance:



In [13]:
optimizer = Optimizer_SGD_With_Decay(Float32(0.1), Float32(0.001))
# At this point:
# optimizer.learning_rate = 0.1
# optimizer.decay = 0.001
# optimizer.iterations = 0
# optimizer.current_learning_rate = 0.1

Optimizer_SGD_With_Decay(0.1f0, 0.001f0, 0, 0.1f0)



**Scenario 1: Calling `pre_update_parameters` for the first time (Epoch 1, `iterations` = 0)**

When `pre_update_parameters(optimizer)` is called:
*   `optimizer.decay` (`0.001`) is greater than `0`.
*   The calculation becomes: `0.1 / (1.0 + 0.001 * 0)`
*   This simplifies to: `0.1 / (1.0 + 0.0)` = `0.1 / 1.0` = `0.1`
*   So, `optimizer.current_learning_rate` is updated to `0.1`.

This means for the very first update, the full initial learning rate is used, as no decay has occurred yet.

**Scenario 2: Calling `pre_update_parameters` after 99 iterations (Epoch 100, `iterations` = 99)**

Let's say `optimizer.iterations` has been incremented to `99` by previous `post_update_parameters` calls.
When `pre_update_parameters(optimizer)` is called:
*   `optimizer.decay` (`0.001`) is greater than `0`.
*   The calculation becomes: `0.1 / (1.0 + 0.001 * 99)`
*   This simplifies to: `0.1 / (1.0 + 0.099)` = `0.1 / 1.099` $\approx$ `0.09099`
*   So, `optimizer.current_learning_rate` is updated to approximately `0.09099`.

As you can see, the learning rate has slightly decreased from `0.1` to `0.09099` due to the decay over 99 iterations. This smaller learning rate will be used for the parameter updates in this specific epoch.

---

### 2. `post_update_parameters` Function

This function simply increments the `iterations` counter. It should be called *after* the actual weight and bias updates in each training step.



In [14]:
function post_update_parameters(optimizer::Optimizer_SGD_With_Decay)
    optimizer.iterations += 1
end

post_update_parameters (generic function with 1 method)



**Explanation with Mock Data for `post_update_parameters`:**

Using the same `optimizer` instance:



In [15]:
optimizer = Optimizer_SGD_With_Decay(Float32(0.1), Float32(0.001))
# Initially: optimizer.iterations = 0

Optimizer_SGD_With_Decay(0.1f0, 0.001f0, 0, 0.1f0)



**Scenario 1: Calling `post_update_parameters` after Epoch 1's updates**

After the weights and biases for Epoch 1 have been updated using the `current_learning_rate` (which was `0.1`):
When `post_update_parameters(optimizer)` is called:
*   `optimizer.iterations` changes from `0` to `1`.

**Scenario 2: Calling `post_update_parameters` after Epoch 2's updates**

After the weights and biases for Epoch 2 have been updated (using the `current_learning_rate` calculated when `iterations` was `1`):
When `post_update_parameters(optimizer)` is called:
*   `optimizer.iterations` changes from `1` to `2`.

This function ensures that the `iterations` counter correctly reflects the number of completed training steps, which is crucial for the `pre_update_parameters` function to calculate the decaying learning rate accurately for the *next* epoch.

---

### 3. Modified `update_parameters` Function

Now, we need to adjust the `update_parameters` function to use the `current_learning_rate` that `pre_update_parameters` has calculated.



n Julia, the dot (.) before an operator like .+, .*, or .+= is used to indicate element-wise operations. This is part of Julia's broadcasting mechanism, which allows operations to be applied across arrays or matrices element by element.

In [16]:
function update_parameters(optimizer::Optimizer_SGD_With_Decay, layer::Layer_dense)
    # Use the dynamically calculated current_learning_rate for updates
    layer.weights .+= -optimizer.current_learning_rate .* layer.dweights
    layer.biases .+= -optimizer.current_learning_rate .* layer.dbiases
end

update_parameters (generic function with 2 methods)



**Explanation with Mock Data for Modified `update_parameters`:**

Let's say `pre_update_parameters` has just been called, and `optimizer.current_learning_rate` is now `0.09099` (as in Scenario 2 for `pre_update_parameters`).

When `update_parameters(optimizer, layer1)` is called:
*   Instead of using the fixed `optimizer.learning_rate` (`0.1`), it now uses `optimizer.current_learning_rate` (`0.09099`).
*   `layer1.weights` will be updated by subtracting `0.09099 * layer1.dweights`.
*   `layer1.biases` will be updated by subtracting `0.09099 * layer1.dbiases`.

This ensures that the decay mechanism is actively applied to the parameter updates, allowing for a more controlled convergence as training progresses.

Now you have all the pieces to integrate this into your training loop!


## Ok lets run this new Optimizer + Decay

In [17]:
# Define the network architecture
layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
layer2 = Layer_dense(64, size(unique(y), 1)) # Number of unique classes in y
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Define the optimizer
optimizer = Optimizer_SGD(Float32(0.5))

println("\n--- Network Architecture Initialized ---")
println("Layer 1 Weights shape: ", size(layer1.weights))
println("Layer 1 Biases shape: ", size(layer1.biases))
println("Layer 2 Weights shape: ", size(layer2.weights))
println("Layer 2 Biases shape: ", size(layer2.biases))
println("Optimizer Learning Rate: ", optimizer.learning_rate)


--- Network Architecture Initialized ---
Layer 1 Weights shape: (2, 64)
Layer 1 Biases shape: (1, 64)
Layer 2 Weights shape: (64, 3)
Layer 2 Biases shape: (1, 3)
Optimizer Learning Rate: 0.5



In [18]:
# --- Training Loop (With new SGD Optimization) ---
println("\n--- Starting Training Loop (With SGD + DECAY Optimization) ---")
println("Note: Weights and biases will now be updated, so loss and accuracy should change.")


--- Starting Training Loop (With SGD + DECAY Optimization) ---
Note: Weights and biases will now be updated, so loss and accuracy should change.


The terms Data Loss (specifically Categorical Cross Entropy Loss) and Accuracy are calculated using distinct mathematical procedures, as described in the sources for evaluating neural network performance in classification problems.

### Data Loss (Categorical Cross Entropy Loss)

Loss is a mathematical function designed to measure how poorly the model performs, with the goal of optimization being to minimize this value.

#### 1. The Core Formula

The loss function used for classification tasks employing a Softmax activation layer is the **Categorical Cross Entropy Loss**.

The fundamental mathematical formula for calculating the loss ($L$) for a single data point is generally given as:

$$L = - \sum (Y_{\text{true}} \times \log(Y_{\text{predicted}}))$$

Where:
*   $Y_{\text{true}}$ represents the **true label** (ground truth), typically in one-hot encoded format.
*   $Y_{\text{predicted}}$ represents the **predicted probability** (confidence) outputted by the Softmax activation function.

#### 2. Calculation Steps for a Single Sample

In classification tasks where true labels are typically **one-hot encoded** (meaning only the correct class index has a value of 1, and all others are 0):

1.  **Identify the Relevant Term:** Because the true label coefficient ($Y_{\text{true}}$) is 1 for the correct class and 0 for all incorrect classes, the terms corresponding to incorrect classes become zero.
2.  **Calculate Loss:** The calculation simplifies to looking only at the predicted confidence of the true class.
    *   The loss is the **negative logarithm of the predicted confidence** for the correct class.
    *   For example, if the true class probability is predicted as $0.7$, the loss is $-\log(0.7)$.
3.  **Ensure Positivity:** The negative sign in the formula is essential because it ensures that the loss value is positive.
4.  **Scaling with Confidence:**
    *   If the predicted confidence for the correct class is high (close to 1.0), $\log(1)$ approaches $0$, resulting in a **low loss** (near zero).
    *   If the predicted confidence for the correct class is low (e.g., $0.1$), $\log(0.1)$ is a high negative value, which, when negated, results in a **high positive loss**.

#### 3. Calculation for a Batch of Data

When calculating the loss for a batch of input data (multiple samples), the individual loss for each sample is computed, and then these individual losses are typically averaged to get the overall cumulative loss for the batch.

### Accuracy

Accuracy is a simpler metric that measures the fraction of correct predictions without considering the numerical confidence level.

#### Calculation Steps:

1.  **Identify Maximum Confidence:** For each input sample (row of the Softmax output), the calculation determines **which class neuron has the highest predicted probability** (maximum confidence).
2.  **Determine Predicted Class:** The index of this maximum probability is taken as the model's prediction for the class.
3.  **Compare to True Label:** This predicted class index is compared directly against the true class index (ground truth label).
4.  **Tally Correct Predictions:** If the predicted index matches the true index, the prediction is counted as correct.
5.  **Final Metric:** Accuracy is the ratio of correct predictions to the total number of samples.

Crucially, **accuracy does not care about the magnitude** of the confidence. If a model predicts the correct class with $0.51$ probability, it is counted as $100\%$ accurate for that sample, just as if it predicted it with $0.99$ probability. Accuracy only tracks whether the highest predicted probability aligns with the actual target class.

In [19]:
n_epochs=10001
# Initialize data_loss and accuracy outside the loop
data_loss::Float32 = Float32(0.0)
accuracy::Float32 = Float32(0.0)

0.0f0

In [20]:
for epoch in 1:n_epochs
    # ---- Forward Pass ----
    forward(layer1,X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss=forward(loss_activation, layer2.output, y)


    # Calculate accuracy
    predictions=[argmax(row) for row in eachrow(loss_activation.output)]  
    accuracy=mean(predictions .==y)


    # ---- Backward Pass ---------
    backward(loss_activation, y)
    backward(layer2, loss_activation.dinputs)
    backward(activation1, layer2.dinputs)
    backward(layer1,activation1.dinputs)

        # --- Update Parameters (Optimization Step) ---
    update_parameters(optimizer, layer1)
    update_parameters(optimizer, layer2)

    # --- Print Results for the Epoch ---
    if epoch == 1 || epoch % 100 == 0 # Print for the first epoch and every 100th epoch
        @printf "Epoch %d: Loss = %.4f, Accuracy = %.4f\n" epoch data_loss accuracy
    end
end

println("\n--- Training Loop Complete ---")
println("Final Loss: ", data_loss)
println("Final Accuracy: ", accuracy)
println("\nNow you can see the loss decreasing and accuracy increasing over epochs!")

Epoch 1: Loss = 1.0986, Accuracy = 0.3067
Epoch 100: Loss = 1.0978, Accuracy = 0.3933
Epoch 200: Loss = 1.0922, Accuracy = 0.4000
Epoch 300: Loss = 1.0812, Accuracy = 0.4200
Epoch 400: Loss = 1.0766, Accuracy = 0.4400
Epoch 500: Loss = 1.0747, Accuracy = 0.4367
Epoch 600: Loss = 1.0734, Accuracy = 0.4500
Epoch 700: Loss = 1.0723, Accuracy = 0.4400
Epoch 800: Loss = 1.0711, Accuracy = 0.4233
Epoch 900: Loss = 1.0698, Accuracy = 0.4267
Epoch 1000: Loss = 1.0687, Accuracy = 0.4300
Epoch 1100: Loss = 1.0678, Accuracy = 0.4267
Epoch 1200: Loss = 1.0667, Accuracy = 0.4367
Epoch 1300: Loss = 1.0654, Accuracy = 0.4467
Epoch 1400: Loss = 1.0637, Accuracy = 0.4500
Epoch 300: Loss = 1.0812, Accuracy = 0.4200
Epoch 400: Loss = 1.0766, Accuracy = 0.4400
Epoch 500: Loss = 1.0747, Accuracy = 0.4367
Epoch 600: Loss = 1.0734, Accuracy = 0.4500
Epoch 700: Loss = 1.0723, Accuracy = 0.4400
Epoch 800: Loss = 1.0711, Accuracy = 0.4233
Epoch 900: Loss = 1.0698, Accuracy = 0.4267
Epoch 1000: Loss = 1.0687, Ac

Of course. Let's break down the training loop line by line using mock data for a simplified scenario.

**Mock Data Setup:**

Imagine we have a tiny dataset with 3 samples and 2 classes.
*   `X` (input features): A `3x2` matrix.
*   `y` (true labels): A vector of length 3. The classes are 1 and 2.



In [22]:
# Mock Data
X = Float32[
    1.0 2.0;  # Sample 1
    2.0 5.0;  # Sample 2
    -1.0 1.0; # Sample 3
]
y = [1, 2, 1] # True labels for Sample 1, 2, 3

3-element Vector{Int64}:
 1
 2
 1



Let's trace what happens inside a single `epoch`.

---

### 1. Forward Pass

The goal of the forward pass is to take the input `X`, pass it through all the layers, and calculate the final loss.



In [23]:
# ---- Forward Pass ----
forward(layer1,X)
forward(activation1, layer1.output)
forward(layer2, activation1.output)
data_loss=forward(loss_activation, layer2.output, y)

10.745397f0



*   **`forward(layer1, X)`**: The input data `X` is passed to the first dense layer. The operation is `output = X * weights + biases`.
    *   **Example**: `X` (`3x2`) is multiplied by `layer1.weights` (`2x64`), resulting in a `3x64` matrix. The `layer1.biases` (`1x64`) are added to each row. The result is stored in `layer1.output`.

*   **`forward(activation1, layer1.output)`**: The output from `layer1` is passed to the ReLU activation function. ReLU replaces all negative values with `0`.
    *   **Example**: If a row in `layer1.output` was `[0.5, -0.2, 1.8, -3.1, ...]`, it becomes `[0.5, 0.0, 1.8, 0.0, ...]`. The result is stored in `activation1.output`.

*   **`forward(layer2, activation1.output)`**: The activated output from `activation1` is passed to the second dense layer.
    *   **Example**: `activation1.output` (`3x64`) is multiplied by `layer2.weights` (`64x2` since we have 2 classes), resulting in a `3x2` matrix. The `layer2.biases` (`1x2`) are added. The result is stored in `layer2.output`. These are the raw model outputs, often called "logits".
    *   Let's say `layer2.output` is now: `Float32[1.2 0.9; 0.8 2.5; 1.5 0.1]`

*   **`data_loss=forward(loss_activation, layer2.output, y)`**: This is a combined step.
    1.  **Softmax**: The logits in `layer2.output` are converted into probabilities.
        *   **Example**: `[1.2 0.9]` becomes something like `[0.57 0.43]`. `[0.8 2.5]` becomes `[0.15 0.85]`. The full probability matrix is stored in `loss_activation.output`.
        *   `loss_activation.output` might be: `Float32[0.57 0.43; 0.15 0.85; 0.80 0.20]`
    2.  **Loss Calculation**: It calculates the categorical cross-entropy loss. It looks at the probabilities the model assigned to the *correct* classes (`y = [1, 2, 1]`) and calculates the average negative log of these probabilities.
        *   **Example**:
            *   Sample 1 (true class 1): Probability was `0.57`. Loss is `-log(0.57)`.
            *   Sample 2 (true class 2): Probability was `0.85`. Loss is `-log(0.85)`.
            *   Sample 3 (true class 1): Probability was `0.80`. Loss is `-log(0.80)`.
        *   `data_loss` is the average of these three values.

---

### 2. Accuracy Calculation

This section checks how many predictions the model got right.



In [24]:
# Calculate accuracy
predictions=[argmax(row) for row in eachrow(loss_activation.output)]  
accuracy=mean(predictions .==y)

0.3333333333333333



*   **`predictions=[argmax(row) ...]`**: For each sample's probability output, `argmax` finds the index (the class) with the highest probability.
    *   **Example**: From `loss_activation.output` `Float32[0.57 0.43; 0.15 0.85; 0.80 0.20]`:
        *   Row 1: `argmax([0.57, 0.43])` is `1`.
        *   Row 2: `argmax([0.15, 0.85])` is `2`.
        *   Row 3: `argmax([0.80, 0.20])` is `1`.
    *   So, `predictions` becomes `[1, 2, 1]`.

*   **`accuracy=mean(predictions .== y)`**: It compares the `predictions` vector with the true labels `y`.
    *   **Example**: `predictions` `[1, 2, 1]` is compared to `y` `[1, 2, 1]`.
    *   The comparison `predictions .== y` results in a boolean array: `[true, true, true]`.
    *   `mean()` of this array treats `true` as `1` and `false` as `0`. The mean is `(1+1+1)/3 = 1.0`. The accuracy for this epoch is 100%.

---

### 3. Backward Pass

The goal here is to calculate the gradients (derivatives) for all weights and biases, indicating how they should be adjusted to reduce the loss.



In [25]:
# ---- Backward Pass ---------
backward(loss_activation, y)
backward(layer2, loss_activation.dinputs)
backward(activation1, layer2.dinputs)
backward(layer1,activation1.dinputs)

3×2 Matrix{Float32}:
   1.427   10.451
   0.0      0.0
 -11.1511  18.859



*   **`backward(loss_activation, y)`**: Calculates the initial gradient. This is the derivative of the loss with respect to the output of `layer2`. The result is stored in `loss_activation.dinputs`.
*   **`backward(layer2, loss_activation.dinputs)`**: Takes the upstream gradient and calculates the gradients for `layer2`'s weights and biases (`layer2.dweights`, `layer2.dbiases`). It also calculates the gradient to be passed back to the previous layer (`layer2.dinputs`).
*   **`backward(activation1, layer2.dinputs)`**: Takes the gradient from `layer2` and passes it through the ReLU activation. The gradient is unchanged for positive inputs but becomes zero for negative inputs from the forward pass. The result is stored in `activation1.dinputs`.
*   **`backward(layer1, activation1.dinputs)`**: Takes the gradient from `activation1` and calculates the gradients for `layer1`'s weights and biases (`layer1.dweights`, `layer1.dbiases`).

---

### 4. Update Parameters

This is where the model learns. It uses the calculated gradients to update the weights and biases.



In [26]:
# --- Update Parameters (Optimization Step) ---
update_parameters(optimizer, layer1)
update_parameters(optimizer, layer2)

1×3 Matrix{Float32}:
 1.2269  -1.26542  0.0385139



*   **`update_parameters(...)`**: This function adjusts the weights and biases using the formula: `parameter = parameter - learning_rate * gradient`.
    *   **Example for `layer1.weights`**:
        *   `layer1.weights` (current value)
        *   `optimizer.learning_rate` (e.g., `0.5`)
        *   `layer1.dweights` (gradient calculated in backward pass)
        *   The new `layer1.weights` will be `layer1.weights - 0.5 * layer1.dweights`. This small adjustment nudges the weights in the direction that will decrease the loss.

---

### 5. Print Results

This section provides feedback on the training progress at specific intervals.



In [27]:
# --- Print Results for the Epoch ---
if epoch == 1 || epoch % 100 == 0
    @printf "Epoch %d: Loss = %.4f, Accuracy = %.4f\n" epoch data_loss accuracy
end

UndefVarError: UndefVarError: `epoch` not defined in `Main`
Suggestion: check for spelling errors or missing imports.



*   **`if epoch == 1 || epoch % 100 == 0`**: This condition is true only for the very first epoch and then for every 100th epoch (100, 200, 300, etc.). This prevents flooding the output with information for every single epoch.
*   **`@printf "..."`**: If the condition is met, this line prints a formatted string showing the current epoch number, the calculated `data_loss`, and the `accuracy`.

Optimization with momentum refers to a technique used to accelerate **gradient descent (GD)** convergence and reduce the number of oscillations that can occur during training neural networks. It is a foundational concept used in modern neural network frameworks, such as the Adam optimizer.

### Intuitive Mechanism

Momentum is introduced to solve problems encountered in simple gradient descent, where a fixed step size can lead to excessive oscillations if the learning rate is too high, or getting stuck in local minima if the learning rate is too low.

The core idea behind momentum is that **the past matters**. Instead of only looking at the current negative gradient direction (steepest descent), momentum incorporates the directions learned in past updates:

1.  **Reducing Oscillations:** When calculating the current gradient update, momentum takes past gradient updates into account.
2.  **Vector Cancellation:** In areas where the function landscape causes zigzag patterns (oscillations), the left and right directional components of the past gradients tend to cancel each other out.
3.  **Smoother Convergence:** The primary component that survives this cancellation is the downward direction (the steepest descent towards the minimum), which nudges the optimization along a much smoother, straighter path.
4.  **Escaping Local Minima:** Momentum provides a small "nudge" that can help the system escape localized regions of low loss (local minima).

### Mathematical Formulation (Weight Update Rule)

When momentum is included, the weight update rule expands beyond traditional gradient descent:

$$
W_{\text{new}} = W_{\text{old}} + \text{Weight Update}
$$

The **Weight Update** is a summation of two terms, allowing the previous update direction to influence the current change:

$$\text{Weight Update} = \left(\text{Momentum Factor} \times \text{Previous Weight Updates}\right) - \left(\alpha \times \frac{\partial L}{\partial W}\right)$$

Where:
*   **$\frac{\partial L}{\partial W}$** represents the **current gradient** (partial derivative of the loss $L$ with respect to the weight $W$). This term is derived from back propagation.
*   **$\alpha$** is the **learning rate** (step size). When used with momentum, this learning rate is often set to decay (decrease) over time as iterations proceed, ensuring exploration at the beginning and precise convergence later.
*   **Momentum Factor:** This is a fixed value (often 0.9 in practice) that governs how much the past weight updates (or "direction of previous changes") influence the current update. A higher momentum factor gives greater weightage to the history.

The previous weight updates are accumulated and stored for each parameter (weights and biases) in objects specific to the layer (e.g., `layer.weight_momentums`).

### Observed Benefits

Implementing gradient descent with momentum yields significant improvements in training performance:

*   **Faster Convergence:** Momentum allows the solution to reach the optimum much faster because oscillations are reduced.
*   **Higher Accuracy:** Experiments show that incorporating momentum leads to substantially higher accuracy and lower loss compared to basic gradient descent or gradient descent with only learning rate decay. For instance, adding momentum increased accuracy from 0.717 (with decay only) to 0.953 in one comparison.
*   **Reduced Stagnation:** Momentum prevents the optimization process from stalling, which can happen when using a low, fixed learning rate or when using other optimizers like Adagrad after many iterations.

 Here is a step-by-step guide on how to implement SGD with momentum, including the necessary code modifications and a detailed explanation with mock data.

### 1. Modify the `Layer_dense` Struct

First, we need to update the `Layer_dense` struct to store the "previous updates" for both weights and biases. These are the momentums.



In [28]:
// ...existing code...
mutable struct Layer_dense
    inputs::Matrix{Float32}
    weights::Matrix{Float32} 
    biases ::Matrix{Float32}
    output::Matrix{Float32} 
    dweights::Matrix{Float32}
    dbiases::Matrix{Float32}
    dinputs::Matrix{Float32}
    # Add momentums
    weight_momentums::Matrix{Float32}
    bias_momentums::Matrix{Float32}

    function Layer_dense(n_inputs::Int, n_neurons::Int)
        # Initialize weights with small random numbers from a Gaussian distribution
        weights = Float32(0.01) * randn(Float32, n_inputs, n_neurons)
        biases = zeros(Float32, 1, n_neurons)
        # Initialize momentums to zeros
        weight_momentums = zeros(Float32, n_inputs, n_neurons)
        bias_momentums = zeros(Float32, 1, n_neurons)
        new(Matrix{Float32}(undef,0,0), weights, biases, Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), Matrix{Float32}(undef,0,0), weight_momentums, bias_momentums)
    end
end

# ...existing code...

Base.Meta.ParseError: ParseError:
# Error @ c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y111sZmlsZQ==.jl:1:1
// ...existing code...
└┘ ── not a unary operator



### 2. Create the `Optimizer_SGD_Momentum` Struct

Next, we define a new optimizer struct that includes a `momentum` parameter.



In [29]:
mutable struct Optimizer_SGD_Momentum
    learning_rate::Float32
    decay::Float32
    iterations::Int
    current_learning_rate::Float32
    momentum::Float32

    function Optimizer_SGD_Momentum(learning_rate::Float32=Float32(1.0), decay::Float32=Float32(0.0), momentum::Float32=Float32(0.0))
        new(learning_rate, decay, 0, learning_rate, momentum)
    end
end



### 3. Implement the Update Logic

We'll create the `pre_update`, `post_update`, and the main `update_parameters` functions for our new optimizer. The core momentum logic resides in `update_parameters`.



In [30]:
function pre_update_parameters(optimizer::Optimizer_SGD_Momentum)
    if optimizer.decay>0
        optimizer.current_learning_rate=optimizer.learning_rate/(Fl)
    
end

Base.Meta.ParseError: ParseError:
# Error @ c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y115sZmlsZQ==.jl:5:4
    
end
#  └ ── Expected `end`



### Explanation with Mock Data

Let's trace the update for a single weight over two iterations to see how momentum works.

**Setup:**
*   `optimizer = Optimizer_SGD_Momentum(learning_rate=0.1, momentum=0.9)`
*   A single weight in `layer.weights` is `1.0`.
*   The corresponding `layer.weight_momentums` starts at `0.0`.

---

**Iteration 1:**

1.  The backward pass calculates a gradient (`dweights`) for our weight, let's say it's `0.5`.
2.  The `update_parameters` function is called.
3.  **Calculate `weight_update`**:
    *   `weight_update = (momentum * previous_momentum) - (learning_rate * gradient)`
    *   `weight_update = (0.9 * 0.0) - (0.1 * 0.5)`
    *   `weight_update = 0.0 - 0.05 = -0.05`
4.  **Store new momentum**: `layer.weight_momentums` is now `-0.05`.
5.  **Update weight**:
    *   `layer.weights = 1.0 + (-0.05) = 0.95`

The weight has been updated based on the current gradient.

---

**Iteration 2:**

1.  The backward pass calculates a new gradient, let's say it's `0.4`.
2.  The `update_parameters` function is called again.
3.  **Calculate `weight_update`**:
    *   This time, `previous_momentum` is not zero! It's `-0.05` from the last step.
    *   `weight_update = (0.9 * -0.05) - (0.1 * 0.4)`
    *   `weight_update = -0.045 - 0.04 = -0.085`
4.  **Store new momentum**: `layer.weight_momentums` is now `-0.085`.
5.  **Update weight**:
    *   `layer.weights = 0.95 + (-0.085) = 0.865`

Notice that the update in Iteration 2 (`-0.085`) was larger than just the gradient part (`-0.04`) because it "remembered" the direction from the previous step. This is how momentum helps accelerate training and smooth out oscillations.

### 4. Integrate into the Training Loop

Finally, you can use this new optimizer in your training loop.


In [31]:
# --- Re-initialize the network and optimizer ---
layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
layer2 = Layer_dense(64, size(unique(y), 1))
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Use the new optimizer with momentum
optimizer = Optimizer_SGD_Momentum(learning_rate=Float32(0.1), decay=Float32(1e-4), momentum=Float32(0.9))

println("\n--- Starting Training Loop (With SGD + Momentum) ---")

n_epochs = 10001
data_loss::Float32 = 0.0
accuracy::Float32 = 0.0

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(layer1, X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss = forward(loss_activation, layer2.output, y)
    
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    backward(loss_activation, y)
    backward(layer2, loss_activation.dinputs)
    backward(activation1, layer2.dinputs)
    backward(layer1, activation1.dinputs)

    # --- Optimization Step ---
    pre_update_parameters(optimizer) # For learning rate decay
    update_parameters(optimizer, layer1)
    update_parameters(optimizer, layer2)
    post_update_parameters(optimizer) # Increment iterations

    # --- Print Results ---
    if epoch == 1 || epoch % 100 == 0
        @printf "Epoch %d: Loss=%.4f, Acc=%.4f, LR=%.4f\n" epoch data_loss accuracy optimizer.current_learning_rate
    end
end

println("\n--- Training Complete ---")

MethodError: MethodError: no method matching Optimizer_SGD_Momentum(; learning_rate::Float32, decay::Float32, momentum::Float32)
This method may not support any kwargs.

Closest candidates are:
  Optimizer_SGD_Momentum(!Matched::Float32, !Matched::Float32, !Matched::Float32) got unsupported keyword arguments "learning_rate", "decay", "momentum"
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y113sZmlsZQ==.jl:8
  Optimizer_SGD_Momentum(!Matched::Float32, !Matched::Float32) got unsupported keyword arguments "learning_rate", "decay", "momentum"
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y113sZmlsZQ==.jl:8
  Optimizer_SGD_Momentum(!Matched::Float32) got unsupported keyword arguments "learning_rate", "decay", "momentum"
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y113sZmlsZQ==.jl:8
  ...


The past matters significantly in optimization procedures, particularly in neural network training using advanced methods like **Momentum** and **Adaptive Gradient Optimizers**, as it helps accelerate convergence and overcome inherent limitations of simple gradient descent.

The importance of the past is realized primarily through two mechanisms:

### 1. Reducing Oscillations and Accelerating Convergence (Momentum)

Momentum is incorporated because the past direction provides valuable information about where the optimization is generally heading.

*   **Influence on Update Direction:** Momentum uses the **previous update direction to influence the next update**. If a standard gradient descent update causes zigzag oscillations (moving back and forth across a valley in the loss landscape), momentum leverages past updates to smooth this path.
*   **Vector Cancellation:** When past gradients are taken into account, the directional components that cause oscillations (e.g., left and right movements) tend to **cancel each other out**, meaning the direction of update favors the overall steepest descent (e.g., the purely downward direction).
*   **Mathematical Representation:** The weight update rule in momentum includes a factor multiplied by the previous weight updates, ensuring that the current update direction is influenced by the entire history of past changes.

### 2. Enabling Adaptive Learning Rates (RMSProp/Adagrad Cache)

The history of gradients also matters for calculating an **adaptive step size** for each individual weight parameter, a concept utilized in Adagrad, RMSProp, and Adam.

*   **Adaptive Step Size:** The core idea is that weights with historically large derivatives should use a smaller effective step size, and weights with historically small derivatives should use a larger effective step size.
*   **Accumulating History (Cache):** Adaptive optimizers maintain a "cache" (historically represented by $G$) for every parameter, which accumulates the **square of past gradients**.
*   **Determining Step Size:** This accumulated history (cache) is placed in the denominator of the parameter update formula (e.g., divided by $\sqrt{\text{cache}} + \text{Epsilon}$). If the accumulated squared gradient values (cache) are high, the effective step size is smaller, achieving the desired adaptive normalization.
*   **Preventing Stagnation (RMSProp/Adam):** In optimizers like RMSProp and Adam, the cache is not just a straightforward accumulation (as in Adagrad, which can lead to learning stagnation if the cache becomes too large). Instead, the cache update weights the **past cache value** heavily (using parameters like $\rho$ or $\beta_2$) compared to the current squared gradient. This weighting solves the Adagrad disadvantage because the cache does not increase indefinitely, preventing the effective learning rate from decaying too quickly and ensuring learning does not stall.

### Adam Optimizer

The Adam Optimizer (Adaptive Moment Estimation) is the most common modern optimizer because it **combines both of these mechanisms**:

1.  It uses a **momentum term** in the numerator (which incorporates past gradients).
2.  It uses an **adaptive cache term** in the denominator (which incorporates past squared gradients) to scale the learning rate for each parameter.

Implementing momentum gradient descent requires defining an optimizer that tracks the history of past weight updates to influence the current direction, thereby reducing oscillations and accelerating convergence.

This implementation typically involves creating an **Optimizer class** that manages the learning rate decay and applies the specific momentum-based update rules to the weights and biases of each layer.

## Conceptual Basis of Momentum

Momentum helps accelerate convergence by leveraging the direction of previous updates.

1.  **Reducing Oscillations:** In traditional gradient descent, weights may oscillate back and forth across a valley in the loss landscape. Momentum takes past updates into account, allowing these oscillations (like "left" and "right" vector components) to cancel each other out. This ensures that the surviving component primarily favors the overall steepest descent (e.g., the purely downward direction), resulting in a much smoother path toward the minimum.
2.  **Past Matters:** Momentum uses the **previous update direction to influence the next update**. The weights update direction depends on the current gradient as well as the entire history of past updates.
3.  **Preventing Stagnation:** Momentum can also give a "small nudge" that helps the optimizer escape shallow local minima where simple gradient descent might get stuck.

## Mathematical Implementation

The update rule for a weight ($W$) includes both the current gradient and the influence of the prior weight updates, stored as a momentum term.

The change in weight (the weight update, $\Delta W$) is calculated as:

$$\Delta W = (\text{Momentum Factor} \times \text{Previous Weight Updates}) - (\text{Current Learning Rate} \times \text{Current Gradient})$$

The new weight ($W_{\text{new}}$) is then found by:

$$W_{\text{new}} = W_{\text{old}} + \Delta W$$

Where:

*   **Momentum Factor** ($\beta_1$ or $\mu$): This value (often around $0.9$ in practice) dictates how much weight is given to the history of changes.
*   **Current Learning Rate ($\alpha$):** This is often a decaying rate, meaning it starts high for exploration and decreases over time (iterations $t$) to ensure smooth convergence (exploitation). The formula for decaying the learning rate ($\alpha$) is typically:
    $$\alpha = \frac{\alpha_0}{1 + \text{DK} \times t}$$
    where $\alpha_0$ is the initial step size and DK is the decay rate.
*   **Current Gradient:** This is the partial derivative of the loss ($L$) with respect to the weight ($W$), $\frac{\partial L}{\partial W}$, which is provided by the layer's backward pass.

## Implementation in Code (Optimizer Class)

To implement this, you define an `Optimizer` class that integrates these rules, often tracking variables like `weight_momentums` (which store the previous updates).

The process involves these main methods within the optimizer class:

### 1. `init` Method

The constructor initializes and tracks key parameters for the training process:

*   **Learning Rate** (`learning_rate`, $\alpha_0$): The initial step size.
*   **Decay Rate** (`decay`, DK): Used to decrease $\alpha$ as iterations increase.
*   **Momentum Factor** (`momentum` or `beta_1`): The coefficient controlling the influence of past updates.
*   **Iterations:** Tracks the number of optimization steps taken ($t$).

### 2. `pre_update_params` Method

Before parameters are updated in a given iteration, the current learning rate is calculated using the decay formula:

$$ \alpha_{\text{current}} = \frac{\text{Initial Learning Rate}}{1 + \text{Decay Rate} \times \text{Iterations}} $$

### 3. `update_params` Method

This method is called for each layer and applies the momentum update rule, accessing the layer's current gradients (`layer.Dweights` and `layer.Dbiases`) and using the tracked momentum objects (`layer.weight_momentums` and `layer.bias_momentums`).

**Steps for weights (the same logic applies to biases):**

1.  **Initialize Momentum Terms (if not done yet):** If the optimizer is using momentum (i.e., the momentum factor is not zero), it initializes the `layer.weight_momentums` to zero arrays matching the dimensions of the weights.
2.  **Calculate New Weight Momentum ($\Delta W$):** The new momentum direction (which is the weight update) is calculated by combining the past momentum with the current scaled gradient:
    $$\Delta W = (\text{Momentum Factor} \times \text{Previous Updates}) - (\alpha_{\text{current}} \times \text{Current Gradient})$$
    The result of this calculation is stored back into `layer.weight_momentums` (as it becomes the "previous update" for the next iteration).
3.  **Apply Update:** The layer's weights are updated by adding the newly calculated update vector:
    $$\text{Layer Weights}_{\text{new}} = \text{Layer Weights}_{\text{old}} + \Delta W. $$

### 4. `post_update_params` Method

After parameter updates, the iteration count is increased by one, ensuring the learning rate is correctly adjusted for the next step.

In [32]:
mutable struct Optimization_SGD_Momentum
    
end

In [33]:
mutable struct Layer_dense2
    weights::Matrix{Float32}
    biases::Matrix{Float32}
    dweights::Matrix{Float32} # Gradients for weights
    dbiases::Matrix{Float32}
end


## Optimizer with decay

In [34]:
mutable struct Op_dcay  
    initial_learning_rate::Float32
    current_learning_rate::Float32
    decay_rate::Float32
    iteration::Float32

    p_dcay(;learning_rate=0.1,decay_rate=0.0)=new(learning_rate,learning_rate, decay_rate,0)
  

    

end


# Constructor to initial the state

In [35]:
    function pre_update_parameters!(optimizer::Op_dcay)
        if optimizer.decay_rate>0
            optimizer.current_learning_rate=optimizer.initial_learning_rate/(1+(optimizer.decay_rate*optimizer.iteration))
        end
    end

pre_update_parameters! (generic function with 2 methods)

In [36]:
function param_update!(optimizer::Op_dcay,layer::Layer_dense)
    layer.weights.+=-optimizer.current_learning_rate.* layer.dweights
    layer.biases+=-optimizer.current_learning_rate.* layer.dbiases
    

end


param_update! (generic function with 1 method)

In [37]:
function post_update_parameters!(optimizer::Op_dcay)
    optimizer.iteration+=1
end

post_update_parameters! (generic function with 1 method)

In [38]:
# Mock test'
mock_layer=Layer_dense2(
    Float32[1.0 2.0 3.0; 4.0 5.0 6.0],
    Float32[0.1 0.1 0.1],
    zeros(Float32, 2, 3),
    zeros(Float32, 1, 3)
)

println("Initial setup complete.\n")

Initial setup complete.



In [39]:
mock_layer.dweights.=Float32(0.5)
mock_layer.dbiases.=Float32(0.2)

1×3 Matrix{Float32}:
 0.2  0.2  0.2

In [40]:
mock_optimizer=Op_dcay(learning_rate=Float32(0.1),decay_rate=(Float32(0.01)))

MethodError: MethodError: no method matching Op_dcay(; learning_rate::Float32, decay_rate::Float32)
The type `Op_dcay` exists, but no method is defined for this combination of argument types when trying to construct it.

In [41]:
    mock_layer.dweights.=Float32(0.5)
    mock_layer.dbiases.=Float32(0.2)
for i in 1:10

    pre_update_parameters!(mock_optimizer)
    
    
    println(" --- iteration no : " , mock_optimizer.iteration)
    println("BEFORE : Weight[1,1] ",mock_layer.weights[1,1])
    param_update!(mock_optimizer,mock_layer)
    println("APPLYING : Weight[1,1] ",mock_optimizer.current_learning_rate)
    post_update_parameters!(mock_optimizer)
    println("After : Weight[1,1] ",mock_layer.weights[1,1])
end



UndefVarError: UndefVarError: `mock_optimizer` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [42]:
    mock_layer.dweights.=Float32(0.5)
    mock_layer.dbiases.=Float32(0.2)
for i in 1:10

    pre_update_parameters!(mock_optimizer)
    
    
    println(" --- iteration no : " , mock_optimizer.iteration)
    println("BEFORE : Weight[1,1] ",mock_layer.weights[1,1])
    param_update!(mock_optimizer,mock_layer)
    println("APPLYING : Weight[1,1] ",mock_optimizer.current_learning_rate)
    post_update_parameters!(mock_optimizer)
    println("After : Weight[1,1] ",mock_layer.weights[1,1])
end



UndefVarError: UndefVarError: `mock_optimizer` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## we will now run the real Network

In [43]:
# X, y = create_data(100, 3) 
Layer1=Layer_dense(size(X,2),64) #------> Layer_dense(no_inputs,no_outputs)
#size(X,2)--> no of columns
activation1=Activation_ReLU()
Layer2=Layer_dense(64,size(unique(y),1))  #- if y = [0, 1, 2, 0, 1], then unique(y) returns [0, 1, 2]
loss_activation=Activation_Softmax_Loss_CategoricalCrossentropy()

#using the Op_dcay optimizer

optimizer=Op_dcay(learning_rate=Float32(0.1),decay_rate=Float32(0.01))



MethodError: MethodError: no method matching Op_dcay(; learning_rate::Float32, decay_rate::Float32)
The type `Op_dcay` exists, but no method is defined for this combination of argument types when trying to construct it.

In [44]:
println("\n ----- Network Architecture Initialized for (Op_dcay) ")
println("Layer 1 Weights shape: ", size(layer1.weights))
println("Layer 1 Biases shape: ", size(layer1.biases))
println("Layer 2 Weights shape: ", size(layer2.weights))
println("Layer 2 Biases shape: ", size(layer2.biases))
println("Optimizer Initial Learning Rate: ", optimizer.initial_learning_rate)
println("Optimizer Decay Rate: ", optimizer.decay_rate)


 ----- Network Architecture Initialized for (Op_dcay) 
Layer 1 Weights shape: (2, 64)
Layer 1 Biases shape: (1, 64)
Layer 2 Weights shape: (64, 2)
Layer 2 Biases shape: (1, 2)


ErrorException: type Optimizer_SGD has no field initial_learning_rate

In [45]:
# X, y = create_data(100, 3) 
println("\n--- Starting Training Loop (With Op_dcay Optimization) ---")

n_epochs = 10000
data_loss::Float32 = 0.0
accuracy::Float32 = 0.0

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(layer1, X)
    forward(activation1, layer1.output)
    forward(layer2, activation1.output)
    data_loss = forward(loss_activation, layer2.output, y)
    
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    backward(loss_activation, y)
    backward(layer2, loss_activation.dinputs)
    backward(activation1, layer2.dinputs)
    backward(layer1, activation1.dinputs)

    # --- Optimization Step ---
    pre_update_parameters!(optimizer) # Calculate current learning rate with decay
    param_update!(optimizer, layer1)   # Update layer1 parameters
    param_update!(optimizer, layer2)   # Update layer2 parameters
    post_update_parameters!(optimizer) # Increment iterations

    # --- Print Results ---
    if epoch == 1 || epoch % 20 == 0
        @printf "Epoch %d: Loss=%.4f, Acc=%.4f, LR=%.6f\n" epoch data_loss accuracy optimizer.current_learning_rate
    end
end

println("\n--- Training Complete (Op_dcay) ---")
println("Final Loss: ", data_loss)
println("Final Accuracy: ", accuracy)
println("Final Learning Rate: ", optimizer.current_learning_rate)
println("\nObserve the loss decreasing and accuracy increasing, with the learning rate decaying over epochs!")


--- Starting Training Loop (With Op_dcay Optimization) ---


MethodError: MethodError: no method matching pre_update_parameters!(::Optimizer_SGD)
The function `pre_update_parameters!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  pre_update_parameters!(!Matched::Op_dcay)
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y131sZmlsZQ==.jl:1
  pre_update_parameters!(!Matched::Optimizer_SGD_Decay_Momentum)
   @ Main c:\Users\Admin\Downloads\NIRBHAY\Neural_Network\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y155sZmlsZQ==.jl:19


Implementing optimization with both **learning rate decay** and **momentum** involves defining an optimizer class that incorporates both features into the parameter update rule.

The combination of learning rate decay and momentum helps accelerate convergence and reduce oscillations in gradient descent. The decay ensures that the learning rate is initially high for exploration of the loss landscape but decreases over time for more precise exploitation near the minimum. Momentum incorporates the direction of previous weight updates to smooth the descent path.

This optimization technique is typically implemented by defining an optimizer class with methods for initialization, pre-update parameter checks, parameter updates, and post-update procedures.

### 1. The Optimization Update Rule

The weight update rule when using both decay and momentum is derived from the standard gradient descent update by introducing two modifying factors: the decaying learning rate ($\alpha$) and the momentum term.

**A. Decaying Learning Rate**
The current learning rate ($\alpha$) is calculated based on the initial learning rate ($\alpha_0$), the decay parameter (DK), and the current number of iterations ($T$):

$$\alpha = \frac{\alpha_0}{1 + \text{DK} \times T}$$

The learning rate is updated before parameters are changed (in the `pre_update_params` method). As the number of iterations ($T$) increases, the learning rate decreases.

**B. Momentum Update**
The weight update ($\Delta W$) uses the momentum factor ($\beta$) to influence the current step based on the previous update direction:

$$\Delta W = \text{Momentum Factor} \times \text{Previous Weight Updates} - \alpha \times \frac{\partial L}{\partial W}$$

Where $\frac{\partial L}{\partial W}$ is the gradient of the loss with respect to the weight. This update calculation is performed for both weights and biases.

### 2. Implementation in an Optimizer Class

An `Optimizer_SGD` class (often used to implement vanilla gradient descent, momentum, and decay) is typically structured to manage these components.

#### A. Initialization (`init` Method)
When the optimizer is initialized, the following parameters are tracked:
1. **Initial Learning Rate ($\alpha_0$):** Set (e.g., $1$).
2. **Decay Rate (DK):** Controls how quickly the learning rate decreases (e.g., $0.001$).
3. **Momentum Factor ($\beta$):** Controls the weight given to past updates (e.g., $0.5$ or $0.9$).
4. **Iterations ($T$):** Tracks the total number of updates, initialized to zero.

#### B. Pre-Update Parameters (`pre_update_params`)
This method is called at the start of each training epoch to calculate the current, adapted learning rate:

*   It calculates the `current_learning_rate` using the decay formula: $\alpha = \alpha_0 / (1 + \text{DK} \times T)$.

#### C. Parameter Update (`update_params`)
This method performs the core gradient descent step for a specific layer, utilizing the calculated `current_learning_rate` and the stored momentum values:

1.  **Check for Momentum:** The method first checks if the momentum parameter is greater than zero.
2.  **Initialize/Access Momentum Storage:** For each layer, arrays must be maintained to store the previous weight and bias updates (`layer.weight_momentums` and `layer.bias_momentums`). If they don't exist yet (first iteration), they are initialized to zero.
3.  **Calculate Weight Update ($\Delta W$):** The update for weights is calculated by combining the momentum term (previous update multiplied by $\beta$) and the current gradient step (current learning rate multiplied by the gradient $\frac{\partial L}{\partial W}$):
    $$\Delta W = (\beta \times \text{layer.weight\_momentums}) - (\alpha \times \text{layer.Dweights})$$
    *Note: $\text{layer.Dweights}$ is the gradient $\frac{\partial L}{\partial W}$ found during the backward pass*.
4.  **Store Momentum:** The newly calculated $\Delta W$ is stored in `layer.weight_momentums` for use in the next iteration.
5.  **Apply Update:** The layer's weights are updated using the calculated direction: $\text{layer.weights} += \Delta W$.
6.  **Bias Updates:** The same procedure (steps 3-5) is repeated for the layer's biases, using $\text{layer.Dbiases}$ and `layer.bias_momentums`.
7.  **Standard Update (if no momentum):** If momentum is set to zero, the weights are updated using standard gradient descent with the decaying learning rate: $\text{layer.weights} -= \alpha \times \text{layer.Dweights}$.

#### D. Post-Update Parameters (`post_update_params`)
After parameters are updated for all layers in the iteration, this method increments the iteration count ($T$):

*   $\text{iterations} += 1$. This ensures the learning rate calculation in the next `pre_update_params` call reflects the accumulated training time.

### Summary of Benefits

Using both decay and momentum together provides enhanced performance:

*   **Momentum** helps reduce oscillations by prioritizing the overall direction of descent determined by past gradients, leading to faster convergence.
*   **Decay** ensures that the step size is large initially to quickly navigate the loss landscape (exploration) and then decreases to finely tune parameters near the minimum (exploitation).

For instance, testing on the spiral dataset showed that incorporating learning rate decay increased accuracy from $57.3\%$ (no decay/momentum) to $71.7\%$, and subsequently adding momentum increased the accuracy further to $83.0\%$ or even $95.3\%$ (with a momentum factor of $0.9$).

In [51]:
    mutable struct Optimization_SGD_Decay_Momentum
        initial_learning_rate::Float32
        current_learning_rate::Float32
        decay_rate::Float32
        iteration::Float32
        momentum::Float32


        Optimization_SGD_Decay_Momentum(;learning_rate=Float32(1.0),decay_rate=Float32(1.0), momentum=Float32(0.0))=new(learning_rate,learning_rate,decay_rate,momentum,0)
        



    end




In [52]:
optimizer=Optimization_SGD_Decay_Momentum()

Optimization_SGD_Decay_Momentum(1.0f0, 1.0f0, 1.0f0, 0.0f0, 0.0f0)

In [53]:
    function pre_update_parameters!(optimizer)
        if optimizer.decay_rate>0
            optimizer.current_learning_rate=optimizer.initial_learning_rate/(1+(optimizer.decay_rate*optimizer.iteration))
        end
    end

pre_update_parameters! (generic function with 3 methods)

In [54]:
# Function to update parameters with momentum and decay
function param_update!(Optimizer, layer::Layer_dense)
    # If momentum is enabled
    if optimizer.momentum > 0 #0.9
        # Calculate weight momentums  
        # [Previous weight_momentums] * momentum(0.9) - current_learning_rate(alpha)[calculated in pre_update_parameters] * dweights (Dl/Dw) stored in 
        layer.weight_momentums = optimizer.momentum .* layer.weight_momentums .- optimizer.current_learning_rate .* layer.dweights
        
        # Calculate bias momentums
        # [Previous bias_momentums] * momentum - current_learning_rate * dbiases
        layer.bias_momentums = optimizer.momentum .* layer.bias_momentums .- optimizer.current_learning_rate .* layer.dbiases
        
        # Update weights and biases  ----> new_weights=old_weights-weight_momentums(past+ present)
        layer.weights .+= layer.weight_momentums
        layer.biases .+= layer.bias_momentums
    else
        # Standard SGD update if no momentum
        layer.weights .+= -optimizer.current_learning_rate .* layer.dweights
        layer.biases .+= -optimizer.current_learning_rate .* layer.dbiases
    end
end

# Function to increment the iteration counter
function post_update_parameters!(optimizer::Optimizer_SGD_Decay_Momentum)
    optimizer.iterations += 1
end

post_update_parameters! (generic function with 2 methods)

In [55]:
# --- Optimizer_SGD_Decay_Momentum (combining Decay and Momentum) ---
mutable struct Optimizer_SGD_Decay_Momentum
    initial_learning_rate::Float32
    current_learning_rate::Float32
    decay_rate::Float32
    momentum::Float32
    iterations::Int

    function Optimizer_SGD_Decay_Momentum(;
        learning_rate::Float32=Float32(1.0),
        decay_rate::Float32=Float32(0.0),
        momentum::Float32=Float32(0.0)
    )
        new(learning_rate, learning_rate, decay_rate, momentum, 0)
    end
end

# Function to calculate the current learning rate with decay
function pre_update_parameters!(optimizer::Optimizer_SGD_Decay_Momentum)
    if optimizer.decay_rate > 0
        optimizer.current_learning_rate = optimizer.initial_learning_rate / (Float32(1.0) + optimizer.decay_rate * optimizer.iterations)
    end
end

# Function to update parameters with momentum and decay
 
        # Standard SGD update if no momentum
        layer.weights .+= -optimizer.current_learning_rate .* layer.dweights
        layer.biases .+= -optimizer.current_learning_rate .* layer.dbiases
    end
end

# Function to increment the iteration counter
function post_update_parameters!(optimizer::Optimizer_SGD_Decay_Momentum)
    optimizer.iterations += 1
end

# --- Example of how to use the new optimizer in a training loop ---

# Re-initialize the network layers (important for a fresh start)
# X, y = create_data(100, 3) # Assuming X and y are already generated
Layer1 = Layer_dense(size(X, 2), 64)
activation1 = Activation_ReLU()
Layer2 = Layer_dense(64, size(unique(y), 1))
loss_activation = Activation_Softmax_Loss_CategoricalCrossentropy()

# Initialize the new optimizer with decay and momentum
optimizer_decay_momentum = Optimizer_SGD_Decay_Momentum(
    learning_rate=Float32(0.1),
    decay_rate=Float32(1e-4), # Example decay rate
    momentum=Float32(0.9)     # Example momentum factor
)

println("\n--- Network Architecture Initialized for (SGD + Decay + Momentum) ---")
println("Layer 1 Weights shape: ", size(Layer1.weights))
println("Layer 1 Biases shape: ", size(Layer1.biases))
println("Layer 2 Weights shape: ", size(Layer2.weights))
println("Layer 2 Biases shape: ", size(Layer2.biases))
println("Optimizer Initial Learning Rate: ", optimizer_decay_momentum.initial_learning_rate)
println("Optimizer Decay Rate: ", optimizer_decay_momentum.decay_rate)
println("Optimizer Momentum: ", optimizer_decay_momentum.momentum)

println("\n--- Starting Training Loop (With SGD + Decay + Momentum Optimization) ---")

n_epochs = 10001
data_loss::Float32 = 0.0
accuracy::Float32 = 0.0

for epoch in 1:n_epochs
    # --- Forward Pass ---
    forward(Layer1, X)
    forward(activation1, Layer1.output)
    forward(Layer2, activation1.output)
    data_loss = forward(loss_activation, Layer2.output, y)
    
    predictions = [argmax(row) for row in eachrow(loss_activation.output)]
    accuracy = mean(predictions .== y)
    
    # --- Backward Pass ---
    backward(loss_activation, y)
    backward(Layer2, loss_activation.dinputs)
    backward(activation1, Layer2.dinputs)
    backward(Layer1, activation1.dinputs)

    # --- Optimization Step ---
    pre_update_parameters!(optimizer_decay_momentum) # Calculate current learning rate with decay
    param_update!(optimizer_decay_momentum, Layer1)   # Update Layer1 parameters with momentum
    param_update!(optimizer_decay_momentum, Layer2)   # Update Layer2 parameters with momentum
    post_update_parameters!(optimizer_decay_momentum) # Increment iterations

    # --- Print Results ---
    if epoch == 1 || epoch % 100 == 0
        @printf "Epoch %d: Loss=%.4f, Acc=%.4f, LR=%.6f\n" epoch data_loss accuracy optimizer_decay_momentum.current_learning_rate
    end
end

println("\n--- Training Complete (SGD + Decay + Momentum) ---")
println("Final Loss: ", data_loss)
println("Final Accuracy: ", accuracy)
println("Final Learning Rate: ", optimizer_decay_momentum.current_learning_rate)
println("\nObserve the loss decreasing and accuracy increasing, with the learning rate decaying over epochs!")

UndefVarError: UndefVarError: `layer` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### OPTIMIZATION WITH DECAY+MOMENTUM+ADAPTIVE GRADIENT 

Adagrad (Adaptive Gradient Algorithm) is an **optimization algorithm** for neural network training. It is noteworthy because it introduced the concept that **different parameters (weights and biases) of a neural network should ideally have different learning rates** for better performance.

Key points about Adagrad:

*   **Adaptive Learning Rates:** Unlike methods like standard gradient descent or momentum, where a common, shared step size (or learning rate, $\alpha$) is used across all weights, Adagrad assigns a different, **individual step size** (or adaptive gradient) to each parameter.
*   **Principle of Adaptation:** Adagrad's goal is to ensure that all parameters are updated on similar scales, preventing some weights from changing too slowly (becoming "dead neurons") while others change too quickly. The underlying principle is: **higher the parameter derivative, smaller should be the effective step size, and smaller the parameter derivative, larger should be the step size**.
*   **Mechanism (Cache):** Adagrad maintains a **cache** (or history) of the previous gradients for every parameter. This cache accumulates the **sum of the squared parameter gradients** from all previous iterations up to the current point.
*   **Update Rule:** The parameter update formula in Adagrad involves dividing the standard gradient descent update by the square root of the cache value (plus a small $\epsilon$ to prevent division by zero).
    $$\text{New Weight} = \text{Old Weight} - \alpha \times \frac{\text{Parameter Gradient}}{\sqrt{\text{Cache} + \epsilon}}$$
    This division ensures that parameters with historically large gradients (resulting in a large cache value) receive a smaller effective step size, and vice versa.
*   **Disadvantage:** A major drawback of Adagrad is that since the cache continuously accumulates the sum of squared gradients, the **cache value continuously increases**. If the number of iterations becomes very high, the cache value can become extremely large, making the effective step size (the denominator) very small. This can cause the **weights to stop updating** entirely, leading to **learning stagnation** or the "dead neuron" problem.

Adagrad's core concept of adaptive learning rates for individual parameters led to the development of other important optimizers, such as the Adam Optimizer.

Adagrad (Adaptive Gradient Algorithm) is implemented by using a technique that assigns an **individual, adaptive learning rate (or effective step size)** to each parameter (weight and bias) of the neural network.

The core implementation involves tracking the **history of squared gradients** for every parameter and using this accumulation to determine the parameter's adaptive step size.

Here is a step-by-step breakdown of how Adagrad is implemented:

### 1. The Core Principle: Adaptive Step Size

The fundamental idea of Adagrad is to ensure that all parameters are updated on similar scales. This is achieved by setting the step size to be **inversely proportional** to the historical magnitude of the parameter's derivatives:

*   **Higher the parameter derivative (gradient), smaller should be the effective step size.**
*   **Smaller the parameter derivative, larger should be the effective step size.**

This mechanism prevents parameters with consistently small gradients from stagnating (becoming "dead neurons") and those with large gradients from changing too quickly.

### 2. Maintaining the Cache (Accumulated Gradients)

Adagrad maintains a running history for every parameter, typically called a **cache**. This cache accumulates the **sum of the squared parameter gradients** from all previous iterations up to the current time point.

Mathematically, the cache ($\text{Cache}_t$) at time $t$ is updated as follows:
$$\text{Cache}_t = \text{Cache}_{t-1} + (\text{Parameter Gradient}_t)^2$$
Where $\text{Cache}_0$ starts at zero.

The gradient is squared in this accumulation process to ensure that when the square root is taken later, the denominator remains a positive value.

### 3. Parameter Update Rule

The cache value is used in the denominator of the standard gradient descent update rule to create the adaptive step size.

The update rule for a parameter (weight $W$) at time $t$ is:
$$\text{New Weight} = \text{Old Weight} - \alpha \times \frac{\text{Parameter Gradient}}{\sqrt{\text{Cache} + \epsilon}}$$
Where:

*   $\alpha$ is the initial learning rate (which may also incorporate decay). In Adagrad, the initial step size $\alpha$ is typically set to one.
*   $\text{Parameter Gradient}$ is the partial derivative of the loss function with respect to the weight ($\frac{\partial L}{\partial W}$) at the current iteration.
*   $\text{Cache}$ is the accumulated sum of squared gradients for that specific parameter.
*   $\epsilon$ (Epsilon) is a very small value (e.g., $10^{-7}$) added to the denominator to **prevent division by zero**, especially in initial iterations when the cache value is zero.

### Summary of Implementation Mechanism

The division by the square root of the accumulated cache ensures the adaptive nature:

1.  If a parameter historically has **high gradients**, its **Cache** value will be large. Dividing by this large number results in a **smaller effective step size**, dampening the updates.
2.  If a parameter historically has **small gradients**, its **Cache** value will be small. Dividing by this smaller number results in a **larger effective step size**, accelerating the updates.

This logic applies equally to updating both **weights and biases** in the neural network.

### Disadvantage of Adagrad

A key consequence of this implementation is that because the cache continuously accumulates positive squared gradients, its value constantly increases. If the number of iterations becomes very large, the cache value can become extremely high, causing the denominator to become so large that the effective step size becomes vanishingly small. This results in **learning stagnation** where weights stop updating altogether, often leading to the "dead neuron" problem.

While the provided sources discuss the intuition, mathematical basis, and disadvantages of the Adagrad optimization algorithm, and extensively cover the implementation of other components (like dense layers, activation functions, and the overall backward pass), they also explicitly provide the **Python code implementation** for the Adagrad Optimizer class.

The implementation involves creating a dedicated `OptimizerAdagrad` class that tracks parameters, updates the learning rate using decay, and, most critically, maintains a cache of squared gradients for adaptive updates.

Here is a comprehensive breakdown of how Adagrad is implemented in code:

### 1. The `OptimizerAdagrad` Class Structure

The Adagrad optimizer is implemented within an `OptimizerAdagrad` class, which typically includes methods for initialization, pre-update checks, parameter updates, and post-update bookkeeping.

### 2. Initialization (`init` Method)

The initialization method tracks several key parameters necessary for Adagrad and learning rate decay:

*   **Learning Rate and Decay:** It tracks the initial learning rate ($\alpha_{0}$), the decay rate (DK), and the current iteration count (T), as the learning rate is decaying. By default, the initial step size ($\alpha_{0}$) for Adagrad is usually set to **one**.
*   **Epsilon ($\epsilon$):** It tracks Epsilon, a very small value (e.g., $10^{-7}$), which is added to the denominator of the update rule to **prevent division by zero**.
*   **Cache Initialization:** Although the cache values are maintained per layer and per parameter (weights and biases), the class structure holds the logic for creating and accessing these caches.

### 3. Updating the Learning Rate (`pre-update params` Method)

Before weights are updated in any given iteration, the current learning rate is calculated using a decay formula. This decaying learning rate ensures that the optimization starts with a higher step size (exploration) and gradually reduces it (exploitation).

The current learning rate is calculated as:
$$\alpha_{\text{current}} = \frac{\alpha_{0}}{1 + (\text{DK} \times T)}$$
Where $\alpha_{0}$ is the initial learning rate, DK is the decay rate, and $T$ is the iteration count.

### 4. Parameter Update Logic (`update params` Method)

This is the central part of the Adagrad implementation, where individual parameters (weights and biases) are updated adaptively.

#### A. Initializing and Updating the Cache

For every layer passed to the update method, the algorithm first accesses or initializes two arrays, `weight cache` and `bias cache`, which are the **same size** as the layer's weights and biases, respectively. These arrays track the history of squared gradients for each parameter.

The cache is updated by adding the square of the current gradient to the previous accumulated cache value. This is performed separately for weights and biases:

*   **Weight Cache Update:** The new weight cache value is the original cache plus the square of the partial derivative of the loss with respect to the weights ($\frac{\partial L}{\partial W}$), which is stored as `layer.dweights`.
    $$\text{New Weight Cache} = \text{Old Weight Cache} + \left(\frac{\partial L}{\partial W}\right)^2$$
*   **Bias Cache Update:** Similarly, the bias cache is updated using the square of the partial derivative of the loss with respect to the biases.

#### B. Calculating Adaptive Updates

Once the cache is updated, it is used to modify the standard gradient descent update for both weights and biases.

The fundamental Adagrad update rule is:
$$\text{New Parameter} = \text{Old Parameter} - \alpha_{\text{current}} \times \frac{\text{Parameter Gradient}}{\sqrt{\text{Cache} + \epsilon}}$$

This is translated into code as follows:

1.  The learning rate ($\alpha_{\text{current}}$) is accessed.
2.  The partial derivative of the loss (gradient) is accessed (e.g., `layer.dweights` or `layer.dbiases`).
3.  The gradient is divided by the square root of the accumulated cache value (plus Epsilon).
4.  The result of this division (the effective step size) is multiplied by the current learning rate and subtracted from the old parameter value to find the new parameter value.

This process ensures that parameters with **historically large gradients** (resulting in a large cache value) receive a **smaller effective step size**, and vice versa, achieving the goal of adaptive, individual learning rates.

### 5. Post-Update Bookkeeping (`post-update params` Method)

After the parameter values have been updated, the iteration count is increased by one. This is crucial because the learning rate for the next iteration depends on this updated count.